# chapterwise PDF Extract on Kaggle

**Dataset:** attach **`chapterwise-models`** (latest version).

**Publish updates from laptop** (when code changes):
```bash
backend\kaggle\publish_chapterwise_dataset.bat
```
Requires `KAGGLE_USERNAME` + `KAGGLE_API_TOKEN` in `backend/.env`.

**Each session:** GPU T4 x2 · Internet ON · set `KAGGLE_API_SECRET` in cell 2 · copy tunnel URL to laptop `backend/.env`.

Stop the notebook when done — GPU hours tick while the session is open.

In [ ]:
# Copy server modules from the attached chapterwise-models dataset
import shutil
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

REQUIRED = (
    "chapterwise_gpu_server.py",
    "llm_server.py",
    "security.py",
    "pdf_extract.py",
)

missing = []
for name in REQUIRED:
    copied = False
    for src in INPUT.rglob(name):
        shutil.copy2(src, WORK / name)
        print(f"Copied {name} <- {src}")
        copied = True
        break
    if not copied:
        missing.append(name)

if missing:
    raise FileNotFoundError(
        "Missing from dataset: " + ", ".join(missing)
        + "\nRe-run publish_chapterwise_dataset.bat on your laptop,"
        + " attach the new dataset version, then re-run this cell."
    )

print("\nWorking dir:", sorted(p.name for p in WORK.iterdir() if p.is_file()))

In [ ]:
import os

os.environ["KAGGLE_API_SECRET"] = "PASTE_YOUR_STRONG_SECRET_HERE"
# Hugging Face token for gated Llama models (Kaggle notebook: Add-ons -> Secrets)
# os.environ["HF_TOKEN"] = "hf_..."
os.environ.setdefault("CHAPTERWISE_LLM_MODEL", "meta-llama/Llama-3.2-1B-Instruct")
os.environ.setdefault("CHAPTERWISE_LLM_PRELOAD", "0")
# Speed tuning (defaults also set inside chapterwise_gpu_server.py)
os.environ.setdefault("CHAPTERWISE_EXTRACT_WORKERS", str(os.cpu_count() or 4))
os.environ.setdefault("CHAPTERWISE_DEFER_IMAGE_B64", "1")
os.environ.setdefault("CHAPTERWISE_SKIP_OVERLAP", "1")
os.environ.setdefault("CHAPTERWISE_USE_PROCESS_POOL", "1")

!python /kaggle/working/chapterwise_gpu_server.py